### [Bước 1] Kiểm tra cấu hình GPU được cấp phát


In [ ]:
!nvidia-smi


Sat Nov  7 02:59:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### [Bước 2] Kết nối Google Drive


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

raw_dataset = '/content/drive/MyDrive/fire'
zip_dataset = '/content/drive/MyDrive/fire_clean.zip'

print(f"Kiểm tra Google Drive:")
print(f"- Thư mục dataset gốc (/MyDrive/fire): {'Đã tìm thấy' if os.path.exists(raw_dataset) else 'Chưa có'}")
print(f"- File nén sạch (/MyDrive/fire_clean.zip): {'Đã có sẵn' if os.path.exists(zip_dataset) else 'Chưa tạo'}")


Mounted at /content/drive
Kiểm tra Google Drive:
- Thư mục dataset gốc (/MyDrive/fire): Đã tìm thấy
- File nén sạch (/MyDrive/fire_clean.zip): Chưa tạo


### [Bước 3] Tải mã nguồn YOLOv5 & Cài đặt môi trường


In [ ]:
%cd /content
!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

# Cài đặt các thư viện cần thiết
%pip install -qr requirements.txt
%pip install -q onnx onnxruntime streamlit pandas matplotlib seaborn

# Tắt bảng hỏi wandb 30 giây
%env WANDB_MODE=disabled

# Vá lỗi tương thích NumPy mới trong utils/plots.py của YOLOv5
!python -c "import re; p='utils/plots.py'; c=open(p, encoding='utf-8').read(); c=c.replace('annotator.im.save(fname)', 'cv2.imwrite(str(fname), np.asarray(annotator.im)[:, :, ::-1]) if not hasattr(annotator.im, \"save\") else annotator.im.save(fname)'); open(p, 'w', encoding='utf-8').write(c)"

import torch
from IPython.display import Image, display
print(f"Setup hoàn tất. PyTorch {torch.__version__} - GPU sẵn sàng: {torch.cuda.is_available()}")


/content
Cloning into 'yolov5'...
remote: Enumerating objects: 18875, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 18875 (delta 351), reused 336 (delta 336), pack-reused 18466 (from 2)
Receiving objects: 100% (18875/18875), 17.70 MiB | 21.07 MiB/s, done.
Resolving deltas: 100% (12885/12885), done.
/content/yolov5
env: WANDB_MODE=disabled
Setup hoàn tất. PyTorch 2.14.1+cu130 - GPU sẵn sàng: True


### [Bước 4] TIỀN XỬ LÝ ẢNH THÔNG MINH (Smart Cache Pipeline)
* **Nếu Drive đã có `fire_clean.zip`:** Giải nén ngay trong và Train ngay lập tức
* **Nếu chưa có:** Tiến hành quét sạch ảnh hỏng, lọc ảnh nhỏ, chuẩn hóa Bounding Box, thêm negative samples từ `MyDrive/fire` Nén và lưu thẳng `fire_clean.zip` lên Drive để các lần sau dùng.


In [ ]:
import os
import glob
import cv2
from PIL import Image as PILImage
from tqdm import tqdm
import shutil

ZIP_CACHE = '/content/drive/MyDrive/fire_clean.zip'
TARGET_DIR = '/content/dataset_clean'

if os.path.exists(ZIP_CACHE):
    print("Đã tìm thấy fire_clean.zip trên Google Drive!")
    print("Đang giải nén dữ liệu sạch trong vài giây...")
    !unzip -q /content/drive/MyDrive/fire_clean.zip -d /content/
    print("Hoàn tất nạp dữ liệu! Bỏ qua toàn bộ bước tiền xử lý, sẵn sàng train ngay lập tức!")
else:
    print("Chưa có file nén trên Drive. Bắt đầu quy trình tiền xử lý dữ liệu lần đầu...")
    SOURCE_DIR = '/content/drive/MyDrive/fire'

    os.makedirs(f"{TARGET_DIR}/train/images", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/train/labels", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/val/images", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/val/labels", exist_ok=True)

    def preprocess_split(split='train', min_size=100):
        img_dir = os.path.join(SOURCE_DIR, split, 'images')
        lbl_dir = os.path.join(SOURCE_DIR, split, 'labels')
        img_files = glob.glob(f"{img_dir}/*.*")
        print(f"\nĐang tiền xử lý tập [{split.upper()}]: Tổng {len(img_files)} ảnh...")

        valid_cnt, corrupt_cnt = 0, 0
        for img_path in tqdm(img_files):
            try:
                with PILImage.open(img_path) as img:
                    img.verify()
                cv_img = cv2.imread(img_path)
                if cv_img is None:
                    corrupt_cnt += 1
                    continue
                h, w = cv_img.shape[:2]
                if h < min_size or w < min_size:
                    corrupt_cnt += 1
                    continue
            except Exception:
                corrupt_cnt += 1
                continue

            base_name = os.path.splitext(os.path.basename(img_path))[0]
            ext = os.path.splitext(img_path)[1].lower()
            if ext not in ['.jpg', '.jpeg', '.png']:
                ext = '.jpg'

            target_img = f"{TARGET_DIR}/{split}/images/{base_name}{ext}"
            target_lbl = f"{TARGET_DIR}/{split}/labels/{base_name}.txt"
            cv2.imwrite(target_img, cv_img)

            src_lbl = os.path.join(lbl_dir, f"{base_name}.txt")
            cleaned_lines = []
            if os.path.exists(src_lbl):
                with open(src_lbl, 'r', encoding='utf-8') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls_id = 0
                            try:
                                x, y, bw, bh = map(float, parts[1:5])
                                x = max(0.0, min(1.0, x))
                                y = max(0.0, min(1.0, y))
                                bw = max(0.0, min(1.0, bw))
                                bh = max(0.0, min(1.0, bh))
                                if bw > 0.001 and bh > 0.001:
                                    cleaned_lines.append(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")
                            except ValueError:
                                continue
            with open(target_lbl, 'w', encoding='utf-8') as f:
                f.writelines(cleaned_lines)
            valid_cnt += 1
        print(f"Hoàn tất [{split}]: {valid_cnt} ảnh hợp lệ (loại {corrupt_cnt} ảnh lỗi).")

    preprocess_split('train')
    preproce ss_split('val')

    print("\nĐang nén và lưu fire_clean.zip vào Google Drive...")
    !zip -r -q /content/drive/MyDrive/fire_clean.zip /content/dataset_clean
    print("Đã lưu fire_clean.zip lên Google Drive thành công! Các lần sau bạn chỉ mất 5s nạp dữ liệu!")


Chưa có file nén trên Drive. Bắt đầu quy trình tiền xử lý dữ liệu lần đầu...

Đang tiền xử lý tập [TRAIN]: Tổng 3527 ảnh...


100%|██████████| 3527/3527 [27:16<00:00,  2.16it/s]


Hoàn tất [train]: 3527 ảnh hợp lệ (loại 0 ảnh lỗi).

Đang tiền xử lý tập [VAL]: Tổng 150 ảnh...


100%|██████████| 150/150 [02:34<00:00,  1.03s/it]


Hoàn tất [val]: 150 ảnh hợp lệ (loại 0 ảnh lỗi).

Đang nén và lưu fire_clean.zip vào Google Drive...
Đã lưu fire_clean.zip lên Google Drive thành công! Các lần sau bạn chỉ mất 5s nạp dữ liệu!


### Chia bộ dữ liệu 70/15/15

In [ ]:
import os
import glob
import shutil
import random

ROOT_DIR = '/content/dataset_clean'

# 1. Gom tất cả các cặp ảnh và nhãn hiện có
all_imgs = glob.glob(f"{ROOT_DIR}/**/images/*.*", recursive=True)
print(f"Tổng số ảnh: {len(all_imgs)} ảnh.")

# Lọc bỏ trùng lặp nếu có
all_pairs = []
for img_path in all_imgs:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    # Tìm file nhãn tương ứng
    dir_name = os.path.dirname(os.path.dirname(img_path))
    lbl_path = os.path.join(dir_name, 'labels', f"{base_name}.txt")
    all_pairs.append((img_path, lbl_path if os.path.exists(lbl_path) else None))

# 2. Xáo trộn ngẫu nhiên
random.seed(42)  # Cố định seed để kết quả minh bạch, tái lập được
random.shuffle(all_pairs)

total = len(all_pairs)
n_train = int(total * 0.70)
n_val = int(total * 0.15)
n_test = total - n_train - n_val  # 15% còn lại

splits = {
    'train': all_pairs[:n_train],
    'val': all_pairs[n_train:n_train + n_val],
    'test': all_pairs[n_train + n_val:]
}

# 3. Tạo lại thư mục tạm để di chuyển file vào
TEMP_DIR = '/content/dataset_70_15_15'
for split_name, pairs in splits.items():
    os.makedirs(f"{TEMP_DIR}/{split_name}/images", exist_ok=True)
    os.makedirs(f"{TEMP_DIR}/{split_name}/labels", exist_ok=True)

    for img_p, lbl_p in pairs:
        shutil.move(img_p, f"{TEMP_DIR}/{split_name}/images/{os.path.basename(img_p)}")
        if lbl_p and os.path.exists(lbl_p):
            shutil.move(lbl_p, f"{TEMP_DIR}/{split_name}/labels/{os.path.basename(lbl_p)}")
        else:
            # Nếu là ảnh nền không nhãn, tạo file nhãn rỗng
            base = os.path.splitext(os.path.basename(img_p))[0]
            with open(f"{TEMP_DIR}/{split_name}/labels/{base}.txt", 'w') as f:
                pass

# Thay thế dataset_clean bằng thư mục đã chia 70/15/15
shutil.rmtree(ROOT_DIR)
shutil.move(TEMP_DIR, ROOT_DIR)

# 4. Cập nhật lại file fire.yaml
yaml_text = """
path: /content/dataset_clean
train: train/images
val: val/images
test: test/images

nc: 1
names: ['fire']
"""
with open('/content/yolov5/fire.yaml', 'w') as f:
    f.write(yaml_text.strip())

print("\nĐÃ CHIA TỶ LỆ 70 / 15 / 15:")
print(f"Tập Train (70%): {len(os.listdir(f'{ROOT_DIR}/train/images'))} ảnh")
print(f"Tập Val   (15%): {len(os.listdir(f'{ROOT_DIR}/val/images'))} ảnh")
print(f"Tập Test  (15%): {len(os.listdir(f'{ROOT_DIR}/test/images'))} ảnh")

# Nén lưu ngay bản 70/15/15 này lên Drive
print("\nĐang nén lưu fire_clean.zip lên Google Drive...")
!zip -r -q /content/drive/MyDrive/fire_clean.zip /content/dataset_clean
print("Hoàn tất lưu Drive")

Tổng số ảnh: 3677 ảnh.

ĐÃ CHIA TỶ LỆ 70 / 15 / 15:
Tập Train (70%): 2573 ảnh
Tập Val   (15%): 551 ảnh
Tập Test  (15%): 553 ảnh

Đang nén lưu fire_clean.zip lên Google Drive...
Hoàn tất lưu Drive
